# CircuitMind Requirements → Architecture → LLM Evaluation

Paste the completed JSON from the Requirement Agent. This notebook passes it to the Architecture Agent, exports React Flow JSON, and asks an LLM judge to evaluate the result.

In [1]:
import json
import os
import sys
from pathlib import Path
import importlib
from copy import deepcopy
from typing import Literal
import gradio as gr
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
_cwd = Path.cwd()
_repo_root = _cwd if (_cwd / "ai_engine").is_dir() else _cwd.parent
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))
import ai_engine.architecture_agent as _architecture_agent
_architecture_agent = importlib.reload(_architecture_agent)
build_architecture = _architecture_agent.build_architecture
ALLOWED_CATEGORIES = _architecture_agent.ALLOWED_CATEGORIES
ALLOWED_INTERFACES = _architecture_agent.ALLOWED_INTERFACES
load_dotenv()

CATEGORY_COLUMNS = {'Power':0,'Input':1,'Sensor':1,'Processing':2,'Security':2,'Clock':2,'Communication':3,'Network':3,'Storage':3,'Memory':3,'Output':4,'Expansion':4}

def validate_graph(graph):
    graph = deepcopy(graph)
    ids = set()
    for node in graph['nodes']:
        node.pop('position', None)
        assert node['id'] not in ids and node['data']['category'] in ALLOWED_CATEGORIES
        ids.add(node['id'])
    for edge in graph['edges']:
        assert edge['source'] in ids and edge['target'] in ids
        assert edge['data']['interface'] in ALLOWED_INTERFACES
    return graph

def layout_graph(graph, horizontal_spacing=320, vertical_spacing=160):
    flow = validate_graph(graph)
    rows = {}
    for node in flow['nodes']:
        column = CATEGORY_COLUMNS.get(node['data']['category'], 2)
        row = rows.get(column, 0)
        rows[column] = row + 1
        node['position'] = {'x':column * horizontal_spacing, 'y':row * vertical_spacing}
    return flow

c:\Rayyan\Rayyan Development\circuit_mind_dev\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## LLM judge and connected pipeline

In [2]:
class ArchitectureJudgement(BaseModel):
    overall_score: int = Field(ge=1, le=5)
    relevance_score: int = Field(ge=1, le=5)
    groundedness_score: int = Field(ge=1, le=5)
    graph_quality_score: int = Field(ge=1, le=5)
    assumption_quality_score: int = Field(ge=1, le=5)
    verdict: Literal['PASS','REVIEW']
    strengths: list[str] = Field(default_factory=list)
    weaknesses: list[str] = Field(default_factory=list)
    warnings: list[str] = Field(default_factory=list)

judge_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a senior embedded-systems architecture reviewer. Judge whether the generated architecture is complete, relevant, vendor-neutral, grounded in the requirements, and suitable for downstream agents. Penalize missing subsystems, unsupported interfaces, exact vendor/IC choices, weak assumptions, and missing design warnings. Return only the structured evaluation.'),
    ('human', 'Requirement Agent JSON:\n{requirements}\n\nArchitecture Agent JSON:\n{architecture}')
])

def judge_chain():
    key = os.getenv('GROQ_API_KEY')
    if not key:
        raise RuntimeError('GROQ_API_KEY is missing. Set it before running evaluation.')
    model = ChatGroq(model=os.getenv('GROQ_MODEL','llama-3.3-70b-versatile'), groq_api_key=key, temperature=0, max_retries=2)
    return judge_prompt | model.with_structured_output(ArchitectureJudgement)

def run_pipeline(requirements):
    if not isinstance(requirements, dict) or not requirements:
        raise ValueError('Requirement Agent JSON must be a non-empty object.')
    agent_result = build_architecture(requirements)
    canonical = validate_graph(agent_result.get('architecture_graph', agent_result.get('react_flow')))
    react_flow = layout_graph(canonical)
    architecture_result = dict(agent_result, architecture_graph=canonical, react_flow=react_flow)
    judgement = judge_chain().invoke({'requirements':json.dumps(requirements, indent=2), 'architecture':json.dumps(architecture_result, indent=2)})
    evaluation = judgement.model_dump()
    evaluation.update({'project_name':requirements.get('project_name','Unnamed project'),'node_count':len(canonical['nodes']),'edge_count':len(canonical['edges'])})
    return architecture_result, evaluation

# This variable represents the completed JSON generated by the Requirement Agent.
# Replace this example with the actual Requirement Agent output. It may be a dict or a JSON string.
# For a Pydantic requirement result, use: requirement_agent_output = result.requirements.model_dump()
# For a JSON response string, use: requirement_agent_output = json.loads(requirement_response_text)
requirement_agent_output = {
    'project_name':'Smart Glasses',
    'objective':'Navigation and voice control',
    'functional_requirements':['Navigation','Voice Control'],
    'hardware_inputs':['Touch Input','Microphone'],
    'hardware_outputs':['Micro OLED Display','Speaker'],
    'connectivity':['Bluetooth'],
    'power_requirements':'Rechargeable Battery',
}

# Normalize and store the handoff consumed by the Architecture Agent.
requirements_json = json.loads(requirement_agent_output) if isinstance(requirement_agent_output, str) else requirement_agent_output
architecture_result, evaluation = run_pipeline(requirements_json)
print(json.dumps(evaluation, indent=2))

{
  "overall_score": 5,
  "relevance_score": 5,
  "groundedness_score": 5,
  "graph_quality_score": 5,
  "assumption_quality_score": 4,
  "verdict": "PASS",
  "strengths": [
    "The architecture is well-structured and easy to follow",
    "The use of inferred subsystems is appropriate"
  ],
  "weaknesses": [
    "Some components, such as the Wireless MCU, are not fully specified"
  ],
  "warnings": [],
  "project_name": "Smart Glasses",
  "node_count": 9,
  "edge_count": 14
}


## Gradio evaluation dashboard

In [ ]:
def dashboard_pipeline(requirements):
    architecture, evaluation = run_pipeline(requirements)
    summary = f"### {evaluation['project_name']} — {evaluation['verdict']}\n\n**Overall LLM score:** {evaluation['overall_score']}/5"
    return summary, architecture, evaluation

with gr.Blocks(title='dunkai Requirements to Architecture Evaluation') as dashboard:
    gr.Markdown('# Requirement Agent → Architecture Agent → LLM Judge')
    gr.Markdown('Paste the completed JSON output from the Requirement Agent. The pipeline passes it directly into the Architecture Agent and sends the resulting architecture to the LLM judge.')
    input_json = gr.JSON(value=requirements_json, label='Requirement Agent output JSON', height=300)
    run_button = gr.Button('Run connected pipeline', variant='primary')
    summary = gr.Markdown()
    with gr.Row():
        architecture_output = gr.JSON(label='Architecture Agent / React Flow output', height=600)
        evaluation_output = gr.JSON(label='LLM evaluation', height=600)
    run_button.click(dashboard_pipeline, inputs=input_json, outputs=[summary, architecture_output, evaluation_output])
dashboard.launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
